In [220]:
################ Import necessary libraries

%pip install pyblp
%pip install statsmodels
%pip install linearmodels
%pip install stargazer

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS
from linearmodels import IV2SLS
from stargazer.stargazer import Stargazer
import matplotlib.pyplot as plt
import pickle


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [258]:
################ Import data
df = pd.read_csv('model_ready.csv')
supply_df = pd.read_csv('supply_ready.csv')

# 将 supply_df 中的 road_fuel_IV 合并到 df
# 假设合并键为 province 和 year

df = df.merge(
    supply_df[['province', 'year', 'road_fuel_IV', 'num_models_in_market']],
    on=['province', 'year'],
    how='left'
)

# 计算 log(sjm), log(s0m), log(sj/g)
df['log_sjm'] = np.log(df['shares'])
df['log_s0m'] = np.log(1 - df.groupby('market_ids')['shares'].transform('sum'))
df['log_sj_g'] = np.log(df['shares'] / df.groupby(['market_ids', 'nesting_ids'])['shares'].transform('sum'))
df['log_charging_stock'] = np.log(df['charging_stations_stock'])

# Set range, log_charging_stock, log_charging_IV, and battery_capacity to 0 for non-EVs
df.loc[df['is_electric'] == 0, ['range', 'battery_capacity']] = 0

In [261]:
# Define list of IVs
iv_list = [
    'cost_shifter', 
    'product_set_size',
    'euclidean_range',
    'local_range',
    'euclidean_battery',
    'local_battery',
    'euclidean_power',
    'local_power',
    'road_fuel_IV',
    'num_models_in_market',
]

iv_str = ' + '.join(iv_list)

In [262]:
################ 2SLS model using custom instruments

# First stage for charging station: regress log_charging_stock on instrument variables
X = sm.add_constant(df[['log_charging_IV']])
y = df['log_charging_stock']
first_stage = sm.OLS(y, X).fit()
df['log_charging_stock_hat'] = first_stage.predict(X)

# Define the formula for the IV2SLS model
formula = f'''
(log_sjm - log_s0m) ~ 0 + is_electric*log_charging_stock_hat + range + power + battery_capacity
    + [net_prices + log_sj_g ~ {iv_str}]
'''
# CAN HAVE HORSEPOWER = POWER / MASS #

iv_model = IV2SLS.from_formula(formula, data=df).fit(cov_type="clustered", clusters=df['market_ids'])


print(iv_model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                log_sjm   R-squared:                      0.9823
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9823
No. Observations:               33545   F-statistic:                 7.239e+04
Date:                Thu, Jul 31 2025   P-value (F-stat)                0.0000
Time:                        03:07:20   Distribution:                  chi2(8)
Cov. Estimator:             clustered                                         
                                                                              
                                         Parameter Estimates                                          
                                    Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------------------------------
is_electric                           -15.658     0.6756   

In [263]:
# Check first stage results
print(iv_model.first_stage.summary)

                First Stage Estimation Results                
                                       net_prices     log_sj_g
--------------------------------------------------------------
R-squared                                  0.8391       0.8954
Partial R-squared                          0.1003       0.0679
Shea's R-squared                           0.1017       0.0689
Partial F-statistic                        1006.9       412.03
P-value (Partial F-stat)                   0.0000       0.0000
Partial F-stat Distn                     chi2(10)     chi2(10)
==================================== ============ ============
is_electric                               -15.596      -3.3693
                                        (-9.3561)    (-9.5514)
log_charging_stock_hat                    -0.6726      -0.2710
                                        (-5.0821)    (-10.766)
range                                      0.0082       0.0028
                                         (6.5088)     (

In [264]:
################ Supply side model

# Create a time trend variable
supply_df['time_trend'] = supply_df['year'].astype(int) - supply_df['year'].astype(int).min() + 1

# Add time_trend
formula = 'log(charging_stations_stock) ~ [log(EV_stock) ~ road_fuel_IV + num_models_in_market + sales_weighted_avg_range] + sub_fix + sub_ope + C(province) + time_trend'

supply_model = IV2SLS.from_formula(formula, data=supply_df).fit()
print(supply_model.summary)
print(supply_model.first_stage.summary)

                               IV-2SLS Estimation Summary                               
Dep. Variable:     log(charging_stations_stock)   R-squared:                      0.9734
Estimator:                              IV-2SLS   Adj. R-squared:                 0.9659
No. Observations:                           155   F-statistic:                 9.753e+05
Date:                          Thu, Jul 31 2025   P-value (F-stat)                0.0000
Time:                                  03:07:28   Distribution:                 chi2(35)
Cov. Estimator:                          robust                                         
                                                                                        
                                   Parameter Estimates                                   
                       Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------
sub_fix           

In [265]:
# Export model summary as LaTeX
# Stargazer does not support different covariate orders for each model,
# so the best practice is to include all you want to show and accept blanks for the other.
main_vars = ['sub_fix', 'sub_ope', 'time_trend', 'log(EV_stock)']

stargazer = Stargazer([supply_model])
stargazer.covariate_order(main_vars)

with open('supply_model_stargazer.tex', 'w', encoding='utf-8') as f:
    f.write(stargazer.render_latex())


In [266]:
# Save demand model results
with open('demand_model_results.pkl', 'wb') as f:
    pickle.dump(iv_model, f)

# Save supply model results
with open('charging_station_model_results.pkl', 'wb') as f:
    pickle.dump(supply_model, f)

In [267]:
# Export model data for counterfactual analysis
df.to_csv('demand_counterfactual.csv', index=False)
supply_df.to_csv('supply_counterfactual.csv', index=False)

In [289]:
def compute_nested_logit_elasticities(
    df,
    demand_par,
    price_col='net_prices',
    market_col='market_ids',
    model_col='model_id',
    nest_col='nesting_ids'
):
    """
    Compute own and cross price elasticities for each market using your equations 12, 14, 15.
    Returns:
        dict: {market_id: elasticity_matrix (DataFrame, index/columns=model_id)}
    """
    alpha = -demand_par.params['net_prices']
    sigma = demand_par.params['log_sj_g']
    elasticity_matrices = {}

    # Compute within-nest shares for all rows
    df['nest_sum'] = df.groupby([market_col, nest_col])['shares'].transform('sum')
    df['within_nest_share'] = df['shares'] / df['nest_sum']

    unique_markets = df[market_col].unique()
    for market_id in unique_markets:
        df_mkt = df[df[market_col] == market_id].copy()
        n = len(df_mkt)
        elasticity = np.zeros((n, n))
        prices = df_mkt[price_col].values
        shares = df_mkt['shares'].values
        within_nest_shares = df_mkt['within_nest_share'].values
        nests = df_mkt[nest_col].values
        model_ids = df_mkt[model_col].values

        # Compute group shares for each product
        group_shares = df_mkt['nest_sum'].values

        for j in range(n):
            s_j = shares[j]
            s_jg = within_nest_shares[j]
            s_g = group_shares[j]
            for k in range(n):
                s_k = shares[k]
                if j == k:
                    # Own-price elasticity (Equation 12)
                    deriv = (1/(1-sigma)) * s_j * (1 - sigma*s_jg - (1-sigma)*s_j)
                elif nests[j] == nests[k]:
                    # Cross-price elasticity within the same group (Equation 14)
                    deriv = -s_j * s_k * (1 + (sigma/(1-sigma)) * (1/s_g))
                else:
                    # Cross-price elasticity across groups (Equation 15)
                    deriv = -s_j * s_k
                # Convert to price elasticity
                elasticity[j, k] = -alpha * prices[k] * deriv / s_j

        elasticity_df = pd.DataFrame(elasticity, index=model_ids, columns=model_ids)
        elasticity_matrices[market_id] = elasticity_df

    return elasticity_matrices

In [290]:
# Demand model results
with open('demand_model_results.pkl', 'rb') as f:
     demand_param = pickle.load(f)

# Charging station model results
with open('charging_station_model_results.pkl', 'rb') as f:
     charging_param = pickle.load(f)

In [291]:
# Compute elasticities with network effects
elasticity_matrices = compute_nested_logit_elasticities(
    df,
    demand_param,
    price_col='net_prices',
    market_col='market_ids',
    model_col='model_id',
    nest_col='nesting_ids'
)

In [292]:
# Compute charging station semi-elasticities (γ_j)
def compute_gamma_dict(beta_N, df, nest_col='nesting_ids', market_col='market_ids', share_col='shares', sigma=None):
    """
    Compute charging station semi-elasticities (γ_j) for all products, returned as a dict by market.
    Returns:
        dict: {market_id: gamma vector for that market}
    """
    gamma_dict = {}
    for market_id in df[market_col].unique():
        market_df = df[df[market_col] == market_id].copy()
        market_df['nest_sum'] = market_df.groupby([market_col, nest_col])[share_col].transform('sum')
        market_df['within_nest_share'] = market_df[share_col] / market_df['nest_sum']
        if isinstance(sigma, dict):
            sigma_g = market_df[nest_col].map(sigma)
        else:
            sigma_g = sigma
        term = (1 / (1 - sigma_g)) * (1 - sigma_g * market_df['within_nest_share'] - (1 - sigma_g) * market_df[share_col])
        gamma = beta_N * market_df[share_col] * term
        gamma_dict[market_id] = gamma.values
    return gamma_dict

In [293]:
# Elasticity with network effects
def compute_total_derivatives(df, eta_dict, gamma_dict, v_2, is_ev_col='is_electric', market_col='market_ids'):
    """
    Compute total derivatives of market shares w.r.t. prices (with network effects).
    Args:
        df (pd.DataFrame): DataFrame with all markets.
        eta_dict (dict): {market_id: JxJ eta matrix for that market}.
        gamma_dict (dict): {market_id: gamma vector for that market}.
        v_2 (float): Sensitivity of charging stations to EV sales.
        is_ev_col (str): Column name for EV indicator.
        market_col (str): Column name for market IDs.
    Returns:
        dict: {market_id: JxJ matrix of total derivatives for each market}.
    """
    total_derivatives_dict = {}
    for market_id in df[market_col].unique():
        market_df = df[df[market_col] == market_id].reset_index(drop=True)
        eta = eta_dict[market_id]
        gamma = gamma_dict[market_id]
        is_ev = market_df[is_ev_col].values.astype(bool)
        shares = market_df['shares'].values
        J = len(gamma)
        total_derivatives = np.zeros((J, J))
        s_ev = np.sum(shares[is_ev])
        sum_gamma_ev = np.sum(gamma[is_ev])
        eta = np.asarray(eta)  
        for j in range(J):
            for k in range(J):
                if is_ev[j]:
                    feedback_term = 0
                    denom = s_ev - v_2 * sum_gamma_ev
                    if denom != 0:
                        feedback_term = v_2 * gamma[j] * np.sum(eta[is_ev, k]) / denom
                    total_derivatives[j, k] = eta[j, k] + feedback_term
                else:
                    total_derivatives[j, k] = eta[j, k]
        total_derivatives_dict[market_id] = total_derivatives
    return total_derivatives_dict

In [294]:
# Calculate gamma_dict for all markets
beta_N = demand_param.params['log_charging_stock_hat']
sigma = demand_param.params['log_sj_g']

gamma_dict = compute_gamma_dict(
    beta_N=beta_N,
    df=df,
    nest_col='nesting_ids',
    market_col='market_ids',
    share_col='shares',
    sigma=sigma
)

# Set your network effect parameter
v_2 = 0.1  # Replace with your estimated value

# Compute total derivatives using elasticity_matrices as eta_dict and gamma_dict as input
total_derivatives_dict = compute_total_derivatives(
    df=df,
    eta_dict=elasticity_matrices,
    gamma_dict=gamma_dict,
    v_2=v_2,
    is_ev_col='is_electric',
    market_col='market_ids'
)


In [295]:
# Show example elasticities with and without network effects

# Pick a sample market
sample_market = list(elasticity_matrices.keys())[0]
print(f"Sample market_id: {sample_market}")

# --- Elasticities WITHOUT network effects ---
sample_elasticity = elasticity_matrices[sample_market]
model_names = df[df['market_ids'] == sample_market]['model'].values
sample_elasticity.index = model_names
sample_elasticity.columns = model_names
print("Elasticity matrix WITHOUT network effects (first 5 models):")
display(sample_elasticity.iloc[:5, :5])

# --- Elasticities WITH network effects ---
sample_total_deriv = total_derivatives_dict[sample_market]
# Convert to DataFrame for display, using same model names
sample_total_deriv_df = pd.DataFrame(sample_total_deriv, index=model_names, columns=model_names)
print("Elasticity matrix WITH network effects (first 5 models):")
display(sample_total_deriv_df.iloc[:5, :5])

Sample market_id: P01Y2019
Elasticity matrix WITHOUT network effects (first 5 models):


,景逸S50,云度π1,云度π3,嘉际,帝豪
景逸S50,-4.367459e+00,2.746760e-04,0.040292,0.000056,2.859822e-03
云度π1,3.575374e-04,-3.355275e+00,0.040292,0.000056,2.859822e-03
云度π3,3.575374e-04,2.746760e-04,-6.522601,0.000056,2.859822e-03
嘉际,8.646972e-08,6.642987e-08,0.000010,-6.703359,6.916423e-07
帝豪,3.575374e-04,2.746760e-04,0.040292,0.000056,-5.819924e+00


Elasticity matrix WITH network effects (first 5 models):


,景逸S50,云度π1,云度π3,嘉际,帝豪
景逸S50,-4.367389,0.000329,0.040385,0.000089,0.002953
云度π1,0.000428,-3.355221,0.040385,0.000089,0.002953
云度π3,0.005610,0.004310,-6.515618,0.002468,0.009808
嘉际,0.027333,0.020998,0.036344,-6.690808,0.036155
帝豪,0.000780,0.000599,0.040853,0.000250,-5.819365


In [284]:
# For a sample market, count how many own-price elasticities are > 1 in absolute value for EVs,
# and show total number of EV models in that market

sample_market = list(elasticity_matrices.keys())[0]
market_df = df[df['market_ids'] == sample_market].reset_index(drop=True)
is_ev = market_df['is_electric'].values.astype(bool)
own_elasticities = np.diag(elasticity_matrices[sample_market].values)

num_ev_above_1 = np.sum(np.abs(own_elasticities[is_ev]) > 1)
num_ev_models = np.sum(is_ev)

print(f"Sample market_id: {sample_market}")
print(f"Number of EV own-price elasticities with |elasticity| > 1: {num_ev_above_1}")
print(f"Total number of EV models in this market: {num_ev_models}")

Sample market_id: P01Y2019
Number of EV own-price elasticities with |elasticity| > 1: 26
Total number of EV models in this market: 35


In [287]:
# For a sample market, count positive cross-price elasticities WITH and WITHOUT network effects

sample_market = list(elasticity_matrices.keys())[0]

# --- WITH network effects ---
sample_total_deriv = total_derivatives_dict[sample_market]
cross_price_mask = ~np.eye(sample_total_deriv.shape[0], dtype=bool)
num_positive_cross_with = np.sum(sample_total_deriv[cross_price_mask] > 0)
print(f"Number of positive cross-price elasticities WITH network effects: {num_positive_cross_with}")

# --- WITHOUT network effects ---
sample_elasticity = elasticity_matrices[sample_market].values
num_positive_cross_without = np.sum(sample_elasticity[cross_price_mask] > 0)
print(f"Number of positive cross-price elasticities WITHOUT network effects: {num_positive_cross_without}")

Number of positive cross-price elasticities WITH network effects: 11652
Number of positive cross-price elasticities WITHOUT network effects: 10986
